In [ ]:
pip install pandas pyarrow

In [ ]:
pip install endee

In [1]:
pip show endee

Name: endee
Version: 0.1.13
Summary: Endee is the Next-Generation Vector Database for Scalable, High-Performance AI
Home-page: https://endee.io
Author: Endee Labs
Author-email: dev@endee.io
License: 
Location: /home/debian/latest_VDB/VectorDBBench/venv2/lib/python3.11/site-packages
Requires: httpx, msgpack, numpy, orjson, pydantic, requests
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
base_path = "/home/debian/latest_VDB/VectorDBBench/vectordataset_label"
dataset_folder = "cohere/cohere_medium_1m"

In [3]:
#For checking parquet file contents
import pyarrow.parquet as pq
import os

file_name = "shuffle_train.parquet"  #input

# Build full path
file_path = os.path.join(base_path, dataset_folder, file_name)


parquet_file = pq.ParquetFile(file_path)

# Read only first batch of rows
first_batch = next(parquet_file.iter_batches(batch_size=5))
preview = first_batch.to_pandas()

for col in preview.columns:
    if preview[col].dtype == object and isinstance(preview[col].iloc[0], list):
        preview[col] = preview[col].apply(lambda x: x[:5] if x is not None else x)

print(preview)

       id                                                emb
0  322406  [0.19600096, -0.5270862, -0.29519123, 0.429556...
1  337610  [0.32829463, -0.055560444, -0.06252914, -0.101...
2  714728  [-0.054155815, 0.009554057, 0.24696879, -0.142...
3  354737  [0.2501778, 0.017853737, -0.43795395, 0.522256...
4  290979  [0.3444257, -0.62243223, -0.20940691, -0.08620...


In [4]:
from endee import Endee
client = Endee(token="localtest")
client.set_base_url("http://148.113.54.173:8080/api/v1")

In [5]:
#For checking indexes
for obj in client.list_indexes().get('indexes'):
    print(obj['name'])
    print(obj['total_elements'])
    print('\t')

test_1M_normal_latest_master
1000000
	
test_1M_numfilter_latest_master_stability_vaib
1000000
	
test_1M_numfilter_latest_master_stability_vaib2
200000
	
test_1M_vaib_labelfilter_0503_1
1000000
	
test_1M_vaib_labelfilter_0503_1_adup1
1000000
	
test_1M_vaib_labelfilter_0503_1_adup3
1000000
	
test_1M_vaib_labelfilter_0503_1_adup4
998004
	
test_1M_vaib_labelfilter_0503_1_asyncbackup
0
	
test_1M_vaib_labelfilter_0503_1_duplicate1
1000000
	
test_1M_vaib_labelfilter_reupsertcheck
500000
	
test_1M_vaib_reupsertcheck
300000
	


In [6]:
#give the index name
index_name = "test_1M_vaib_labelfilter_0503_1_adup4"
index = client.get_index(index_name)

In [7]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_0503_1_adup4',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 1000000,
 'precision': 'int16',
 'M': 16}

In [8]:
#For querying
import pyarrow.parquet as pq
import os

#inputs
label_to_query = "label_1p"
top_k = 30

# Mode 1: Single query id
# Mode 2: Find all query ids with recall < 1
mode = 2          # change to 1 or 2
query_id = 457    # only used in mode 1

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
test_parquet_path = os.path.join(dataset_path, "test.parquet")
gt_parquet_path = os.path.join(dataset_path, f"neighbors_labels_{label_to_query}.parquet")

# Read parquet files
test_df = pq.ParquetFile(test_parquet_path).read().to_pandas()
gt_df = pq.ParquetFile(gt_parquet_path).read().to_pandas()

def run_query(query_id):
    query_row = test_df[test_df["id"] == query_id]
    if query_row.empty:
        print(f"Query ID {query_id} not found in test.parquet")
        return None

    query_vector = query_row["emb"].values[0]

    gt_row = gt_df[gt_df["id"] == query_id]
    if gt_row.empty:
        print(f"Query ID {query_id} not found in ground truth file")
        return None

    ground_truth = gt_row["neighbors_id"].values[0][:top_k]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        filter=[{"label": {"$eq": label_to_query}}],
        include_vectors=False,
    )
    returned_ids = [int(r["id"]) for r in results]

    intersection = len(set(returned_ids) & set(ground_truth))
    recall = intersection / len(ground_truth)

    return returned_ids, ground_truth, recall


if mode == 1:
    result = run_query(query_id)
    if result:
        returned_ids, ground_truth, recall = result
        print(f"Query ID: {query_id}")
        print(f"Returned IDs: {','.join(map(str, returned_ids))}")
        print(f"Ground Truth: {','.join(map(str, ground_truth))}")
        print(f"Recall: {recall:.4f}")

elif mode == 2:
    print(f"Finding all query IDs with recall < 1.0 ...\n")
    low_recall_ids = []
    all_recalls = []
    for query_id in test_df["id"].values:
        result = run_query(query_id)
        if result:
            returned_ids, ground_truth, recall = result
            all_recalls.append(recall)
            if recall < 1.0:
                low_recall_ids.append(query_id)
                print(f"Query ID: {query_id}")
                print(f"Returned IDs: {','.join(map(str, returned_ids))}")
                print(f"Ground Truth: {','.join(map(str, ground_truth))}")
                print(f"Recall: {recall:.4f}\n")

    print(f"Total query IDs with recall < 1.0: {len(low_recall_ids)}")
    print(f"IDs: {low_recall_ids}")
    print(f"Average Recall across all {len(all_recalls)} queries: {sum(all_recalls)/len(all_recalls):.4f}")

Finding all query IDs with recall < 1.0 ...

Query ID: 2
Returned IDs: 916915,153590,759005,515871,151336,91663,375948,500463,225501,360986,265421,402047,294599,492005,632708,911260,561483,408751,959763,554056,930465,21239,124402,73355,770379,18830,401384,350257,782199,918389
Ground Truth: 916915,153590,759005,515871,151336,91663,375948,500463,225501,360986,209632,265421,402047,294599,492005,632708,911260,561483,408751,959763,554056,930465,21239,124402,73355,24196,770379,18830,401384,350257
Recall: 0.9333

Query ID: 6
Returned IDs: 235467,945780,366682,809535,755666,392618,18830,109016,912561,878069,9024,393202,612297,916915,87686,913268,652336,331608,734241,23934,653858,12387,982164,695965,381462,266404,882066,120980,335933,6153
Ground Truth: 235467,945780,366682,809535,755666,392618,18830,109016,912561,878069,9024,285884,393202,612297,916915,87686,913268,652336,331608,734241,23934,653858,12387,500463,982164,695965,381462,266404,882066,120980
Recall: 0.9333

Query ID: 9
Returned IDs: 

In [8]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_0503_1_adup4',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 1000000,
 'precision': 'int16',
 'M': 16}

In [8]:
#For deleting
# Delete by label
label_to_delete = "label_50p" #input

result = index.delete_with_filter([{"label": {"$eq": label_to_delete}}])
print(f"Deleted vectors with label: {label_to_delete}")
print(result)

Deleted vectors with label: label_50p
500000 vectors deleted


In [10]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_reupsertcheck',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 1000000,
 'precision': 'int16',
 'M': 16}

In [8]:
#For reinserting delted vectors
import pyarrow.parquet as pq
import pandas as pd
import os

# User inputs
label_to_insert = "label_50p"
batch_size = 1000

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
labels_path = os.path.join(dataset_path, "scalar_labels.parquet")
emb_path = os.path.join(dataset_path, "shuffle_train.parquet")

# Read labels file fully (small file, just id + label)
labels_df = pq.ParquetFile(labels_path).read().to_pandas()

# Filter only the label we need
filtered_labels = labels_df[labels_df["labels"] == label_to_insert]
valid_ids = set(filtered_labels["id"].values)

print(f"Total vectors to insert: {len(valid_ids)}")

# Process shuffle_train in batches to avoid RAM issues
emb_file = pq.ParquetFile(emb_path)
total_inserted = 0

for batch in emb_file.iter_batches(batch_size=batch_size):
    batch_df = batch.to_pandas()
    
    # Keep only rows whose id is in filtered labels
    batch_df = batch_df[batch_df["id"].isin(valid_ids)]
    
    if batch_df.empty:
        continue
    
    # Merge to get labels
    batch_df = batch_df.merge(filtered_labels[["id", "labels"]], on="id")
    
    batch_vectors = []
    for _, row in batch_df.iterrows():
        vector_data = {
            "id": str(row["id"]),
            "vector": row["emb"],
            "meta": {"id": row["id"]},
            "filter": {
                "id": row["id"],
                "label": row["labels"]
            }
        }
        batch_vectors.append(vector_data)
    
    index.upsert(batch_vectors)
    total_inserted += len(batch_vectors)
    print(f"Upserted {len(batch_vectors)} vectors, total so far: {total_inserted}")

print(f"Done! Total inserted: {total_inserted} vectors with label '{label_to_insert}'")

Total vectors to insert: 500000
Upserted 501 vectors, total so far: 501
Upserted 513 vectors, total so far: 1014
Upserted 481 vectors, total so far: 1495
Upserted 513 vectors, total so far: 2008
Upserted 513 vectors, total so far: 2521
Upserted 520 vectors, total so far: 3041
Upserted 504 vectors, total so far: 3545
Upserted 510 vectors, total so far: 4055
Upserted 503 vectors, total so far: 4558
Upserted 480 vectors, total so far: 5038
Upserted 479 vectors, total so far: 5517
Upserted 478 vectors, total so far: 5995
Upserted 491 vectors, total so far: 6486
Upserted 535 vectors, total so far: 7021
Upserted 499 vectors, total so far: 7520
Upserted 477 vectors, total so far: 7997
Upserted 507 vectors, total so far: 8504
Upserted 486 vectors, total so far: 8990
Upserted 501 vectors, total so far: 9491
Upserted 488 vectors, total so far: 9979
Upserted 535 vectors, total so far: 10514
Upserted 514 vectors, total so far: 11028
Upserted 500 vectors, total so far: 11528
Upserted 488 vectors, t

In [11]:
index.describe()

{'name': 'test_1M_vaib_labelfilter_reupsertcheck',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 1000000,
 'precision': 'int16',
 'M': 16}